# Data Analytics

## Hazmin and Vladyslav

```markdown
## OSMI Mental Health in Tech Survey 2014
```

In [ ]:
### Imports & Data Loading
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

plt.style.use("seaborn-v0_8")

DATA_PATH = Path("..") / ".." / "data" / "osmi-raw.csv"
df = pd.read_csv(DATA_PATH)

df.head()
df.info()
df.describe(include='all').transpose()

In [ ]:
### Missing Values Visualization

plt.figure(figsize=(10,4))
sns.heatmap(df.isna(), cbar=False)
plt.title("Missing Values Heatmap ")
plt.show()

In [ ]:
### Data cleaning and preprocessing
df = df.drop(['comments'], axis= 1)
df = df.drop(['state'], axis= 1)
df = df.drop(['Timestamp'], axis= 1)

# Assign default values for each data type
defaultInt = 0
defaultString = 'NaN'
defaultFloat = 0.0

# Create lists according to data types
integerFeatures = ['Age']
stringFeatures = ['Gender', 'Country', 'self_employed', 'family_history', 'treatment', 'work_interfere',
                 'no_employees', 'remote_work', 'tech_company', 'anonymity', 'leave', 'mental_health_consequence',
                 'phys_health_consequence', 'coworkers', 'supervisor', 'mental_health_interview', 'phys_health_interview',
                 'mental_vs_physical', 'obs_consequence', 'benefits', 'care_options', 'wellness_program',
                 'seek_help']
floatFeatures = []

# Cleaning NaN's
for i in df:
    if i in integerFeatures:
        df[i] = df[i].fillna(defaultInt)
    elif i in stringFeatures:
        df[i] = df[i].fillna(defaultString)
    elif i in floatFeatures:
        df[i] = df[i].fillna(defaultFloat)
    else:
        print('Error: Feature %s not recognized.' % i)

In [ ]:
### Age cleaning

# Replace missing age with median() value
df['Age'] = df['Age'].fillna(df['Age'].median())

# Replace values lower than 18 and higher than 110 with the median
median_age = df['Age'].median()
df.loc[df['Age'] < 18, 'Age'] = median_age
df.loc[df['Age'] > 110, 'Age'] = median_age

# Create age ranges
df['age_range'] = pd.cut(df['Age'], [0, 20, 30, 60, 100], labels=["0-20", "21-30", "31-60", "66-100"], include_lowest=True)

In [ ]:
### Gender cleaning

#To the lowercase 
gender = df['Gender'].str.lower().unique()
#print(gender)

#Gender groups creation
male_str = ["male", "m", "male-ish", "maile", "mal", "male (cis)", "make", "male ", "man","msle", "mail", "malr","cis man", "Cis Male", "cis male"]
other_gender_str = ["trans-female", "something kinda male?", "queer/she/they", "non-binary","nah", "all", "enby", "fluid", "genderqueer", "androgyne", "agender", "male leaning androgynous", "guy (-ish) ^_^", "trans woman", "neuter", "female (trans)", "queer", "ostensibly male, unsure what that really means"]           
female_str = ["female", "woman", "cis female", "f", "femake", "female ","cis-female/femme", "female (cis)", "femail"]

for (row, col) in df.iterrows():

    if str.lower(col.Gender) in male_str:
        df['Gender'].replace(to_replace=col.Gender, value='male', inplace=True)

    if str.lower(col.Gender) in female_str:
        df['Gender'].replace(to_replace=col.Gender, value='female', inplace=True)

    if str.lower(col.Gender) in other_gender_str:
        df['Gender'].replace(to_replace=col.Gender, value='other_gender', inplace=True)

#Get rid of bullshit
stk_list = ['A little about you', 'p']
train_df = df[~df['Gender'].isin(stk_list)]

print(train_df['Gender'].unique())

In [ ]:
### Age Distribution Histogram

# Strip whitespace
df['Age'] = df['Age'].astype(str).str.strip()

# Replace invalid entries with NaN
invalid_age = ["", " ", "?", "N/A", "none", "None", "nan"]
df['Age'] = df['Age'].replace(invalid_age, pd.NA)

# Convert to numeric properly
df['Age'] = pd.to_numeric(df['Age'], errors='coerce')

# Remove unrealistic ages
df.loc[(df['Age'] < 10) | (df['Age'] > 100), 'Age'] = pd.NA

df['Age'].dropna().hist(bins=30)